## Analysis & Dimensionality reduction on scalar features

We will select important features and perform pca on these to create 2 scalar_features arrays (selected, selected+pca) and train 2 models

### 1. Insights on data quality, variance & correlation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

from music_analyzer_v2.dataset import MusicDataset
from music_analyzer_v2.utils import SCALAR_FEATURE_NAMES

In [ ]:
RANDOM_SEED = 42
VARIANCE_THRESHOLD = 1e-5
CORRELATION_THRESHOLD = 0.95

In [ ]:
train_dataset = MusicDataset("train")

X = np.stack([
    train_dataset.get_raw_scalar_features(i)
    for i in range(len(train_dataset))
])

X.shape

In [ ]:
targets = train_dataset.get_targets()
genre_targets = targets["genre_targets"]
feeling_targets = targets["feeling_targets"]

Quality check

In [ ]:
print(f"Number of missing values: {np.isnan(X).sum()}")
print(f"Number of invalid values: {np.isinf(X).sum()}")
print(f"All values are fine: {np.isfinite(X).all()}")

Variance

In [ ]:
variances = np.var(X, axis=0)

variance_df = pd.DataFrame({
    "feature": SCALAR_FEATURE_NAMES,
    "variance": variances, 
})

variance_df.sort_values("variance")

In [ ]:
variant_feature_names = variance_df[variance_df["variance"] >= VARIANCE_THRESHOLD]["feature"]
variant_feature_names

In [ ]:
X_variant = X[:, variance_df["variance"] >= VARIANCE_THRESHOLD]

In [ ]:
scaler_original = StandardScaler()
X_scaled = scaler_original.fit_transform(X)

scaler_variant = StandardScaler()
X_variant_scaled = scaler_variant.fit_transform(X_variant)

Correlation Matrix for original and variant data respectively

In [ ]:
corr_or = pd.DataFrame(X_scaled, columns=SCALAR_FEATURE_NAMES).corr()
corr_or

In [ ]:
corr_v = pd.DataFrame(X_variant_scaled, columns=variant_feature_names).corr()
corr_v

### 2. Filter out non variant and highly correlated features

In [ ]:
# correlation matrix has features of interest only on the upper triangle
# it is symmetrical and has 1 for all diagonal
# using np.triu we create an upper triangular matrix of the same shape
# and then we use it as a mask
upper = corr_v.where(
    np.triu(np.ones(corr_v.shape), k=1).astype(bool)
)

columns_to_drop = [
    column 
    for column in upper.columns
    if any(abs(upper[column]) > CORRELATION_THRESHOLD)
]

# We use the UNSCALED VARIANT data
X_reduced = X_variant.drop(columns=columns_to_drop)

features_passed_filter_1 = X_reduced.columns

print(f"Original number of features: {X.shape[1]}")
print(f"Reduced number of features: {X_reduced.shape[1]}")
print(f"Features filtered out: {", ".join(columns_to_drop)}")
print(f"Features kept: {", ".join(features_passed_filter_1)}")

### 3. Supervised feature selection

In [ ]:
genre_mean_mis = []
feeling_mean_mis = []

for i in range(X.shape[1]):
    genre_mean_mi = mutual_info_classif(
        X_reduced,
        genre_targets[:, i],
        random_state=RANDOM_SEED,
    ).mean()
    genre_mean_mis.append(genre_mean_mi)

    feeling_mean_mi = mutual_info_classif(
        X_reduced,
        feeling_targets[:, i],
        random_state=RANDOM_SEED,
    ).mean()
    feeling_mean_mis.append(feeling_mean_mi)



mi_rankings = pd.DataFrame(
    np.stack([genre_mean_mis, feeling_mean_mis]).reshape(-1,1), 
    rows=features_passed_filter_1, 
    columns=["feature", "genre_mean_mi", "feeling_mean_mi"]
)
mi_rankings

We select the appropriate features

In [ ]:
selected_features = [...]
selected_features

In [ ]:
selected_set = set(selected_features)
mask = [feature in selected_set for feature in features_passed_filter_1]

X_selected = X_reduced[:, mask]

### 4. PCA 

In [ ]:
pca = PCA(n_components=0.95, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_selected)